In [1]:
import pandas as pd
import sqlite3

In [2]:
import os
os.getcwd()

'c:\\Users\\marti\\Documents\\GitHub\\2026-10-BILBAO-FT-Data-Science-LauraML\\2-Data_Analysis\\8-BBDD\\SQL\\Teoria\\SQL_Python\\ejercicio'

In [3]:
connection = sqlite3.connect("cervezas.db")
crsr = connection.cursor()
crsr.execute('PRAGMA foreign_keys = ON;');  #Habilitar el soporte de las FOREIGN KEYS en SQLite

In [4]:
#Primero creamos las 4 tablas(Cervezas, Bares, Empleados, Reparto)
query = '''
 CREATE TABLE Cervezas (
     CodC varchar(255) PRIMARY KEY,
     Envase varchar(255),
     Capacidad float,
     Stock int
 );
 '''
crsr.execute(query);

In [5]:
query = '''
 CREATE TABLE Bares (
     CodB varchar(255) PRIMARY KEY,
     Cif varchar(255),
     Nombre varchar(255),
     Localidad varchar(255)
 );
 '''
crsr.execute(query);

In [6]:
query = '''
 CREATE TABLE Empleados (
     CodE varchar(255) PRIMARY KEY,
     Nombre varchar(255),
     Sueldo int
 );
 '''
crsr.execute(query);

In [7]:
#En la tabla Reparto creo las relaciones con las otras tablas mediante FOREIGN KEYS
query = '''
 CREATE TABLE Reparto (
     CodE varchar(255),
     CodB varchar(255),
     CodC varchar(255),
     Fecha DATE,
     Cantidad int,
     FOREIGN KEY (CodE) REFERENCES Empleados(CodE),
     FOREIGN KEY (CodB) REFERENCES Bares(CodB),
     FOREIGN KEY (CodC) REFERENCES Cervezas(CodC)
 )
'''
crsr.execute(query);

In [8]:
#Completo las tablas con datos
query = '''
    INSERT INTO Cervezas (CodC, Envase, Capacidad, Stock)
    VALUES('01', 'Botella', 0.2, 3600),
    ('02', 'Botella', 0.33, 12000),
    ('03', 'Lata', 0.33, 24000),
    ('04', 'Botella', 1.0, 288),
    ('05', 'Barril', 60.0, 30);
'''
crsr.execute(query);

In [9]:
query = '''
    INSERT INTO Bares (CodB, Cif, Nombre, Localidad) 
    VALUES ('001', '11111111X', 'Stop', 'Villa Botijo'),
    ('002', '22222222Y', 'Las Vegas', 'Villa Botijo'),
    ('003', '-', 'Club Social', 'Las Ranas'),
    ('004', '33333333Z', 'Otra Ronda', 'La Esponja');
'''
crsr.execute(query);

In [10]:
query = '''
    INSERT INTO Empleados (CodE, Nombre, Sueldo) 
    VALUES (1, 'Prudencio Caminero', 120000),
    (2, 'Vicente Merario', 110000),
    (3, 'Valentin Siempre', 100000);
'''
crsr.execute(query);

In [11]:
query = '''
    INSERT INTO Reparto (CodE, CodB, CodC, Fecha, Cantidad) 
    VALUES ('1', '001', '01', '2005-10-21', 240),
    ('1', '001', '02', '2005-10-21', 48),
    ('1', '002', '03', '2005-10-22', 60),
    ('1', '004', '05', '2005-10-22', 4),
    ('2', '002', '03', '2005-10-22', 48),
    ('2', '002', '05', '2005-10-23', 2),
    ('2', '004', '01', '2005-10-23', 480),
    ('2', '004', '02', '2005-10-24', 72),
    ('3', '003', '03', '2005-10-24', 48),
    ('3', '003', '04', '2005-10-25', 20);
'''

crsr.execute(query);

In [12]:
#Hacemos commit para guardar los cambios en la base de datos
connection.commit()

In [ ]:
# 1. Obtener el nombre de los empleados que hayan repartido al bar 'Stop' durante la semana del 17 al 23 de octubre de 2005.
query = '''
    SELECT DISTINCT Empleados.Nombre
    FROM Empleados
    JOIN Reparto ON Empleados.CodE = Reparto.CodE
    JOIN Bares ON Reparto.CodB = Bares.CodB
    WHERE Bares.Nombre = 'Stop'
    AND Reparto.Fecha BETWEEN '2005-10-17' AND '2005-10-23'
'''
crsr.execute(query).fetchall()

[('Prudencio Caminero',)]

In [ ]:
# 2. Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo 'Botella' y capacidad inferior a 1 litro, ordenados por localidad.
query = '''
    SELECT DISTINCT Bares.Cif, Bares.Nombre
    FROM Bares
    JOIN Reparto ON Bares.CodB = Reparto.CodB
    JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
    WHERE Cervezas.Envase = 'Botella' AND Cervezas.Capacidad < 1.0
    ORDER BY Bares.Localidad;
'''
crsr.execute(query)
pd.read_sql_query(query, connection)

,Cif,Nombre
0,33333333Z,Otra Ronda
1,11111111X,Stop


In [19]:
# 3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad) realizados por 'Prudencio Caminero'.
query = '''
    SELECT Bares.Nombre AS Nombre_Bar, Cervezas.Envase, Cervezas.Capacidad, Reparto.Fecha, Reparto.Cantidad
    FROM Reparto
    JOIN Empleados ON Reparto.CodE = Empleados.CodE
    JOIN Bares ON Reparto.CodB = Bares.CodB
    JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
    WHERE Empleados.Nombre = 'Prudencio Caminero';
'''
crsr.execute(query)
pd.read_sql_query(query, connection)

,Nombre_Bar,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,2005-10-21,240
1,Stop,Botella,0.33,2005-10-21,48
2,Las Vegas,Lata,0.33,2005-10-22,60
3,Otra Ronda,Barril,60.00,2005-10-22,4


In [20]:
# 4. Obtener los bares a los que se les ha repartido envases de tipo botella y capacidad 0.2 ó 0.33
query = '''
    SELECT DISTINCT Bares.Nombre AS Nombre_Bar
    FROM Bares
    JOIN Reparto ON Bares.CodB = Reparto.CodB
    JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
    WHERE Cervezas.Envase = 'Botella' AND (Cervezas.Capacidad = 0.2 OR Cervezas.Capacidad = 0.33);
'''
crsr.execute(query)
pd.read_sql_query(query, connection)

,Nombre_Bar
0,Stop
1,Otra Ronda


In [ ]:
# 5. Nombre de los empleados que han repartido a los bares 'Stop' y 'Las Vegas' cervezas con envase botella.
query = '''
    SELECT DISTINCT Empleados.Nombre AS Nombre_Empleado
    FROM Empleados
    JOIN Reparto ON Empleados.CodE = Reparto.CodE
    JOIN Bares ON Reparto.CodB = Bares.CodB
    JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
    WHERE Bares.Nombre IN ('Stop', 'Las Vegas') AND Cervezas.Envase = 'Botella';
'''
crsr.execute(query).fetchall()   

[('Prudencio Caminero',)]

In [22]:
# 6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de Villa Botijo
query = '''
    SELECT Empleados.Nombre AS Nombre_Empleado, COUNT(*) AS Numero_Viajes
    FROM Empleados
    JOIN Reparto ON Empleados.CodE = Reparto.CodE
    JOIN Bares ON Reparto.CodB = Bares.CodB
    WHERE Bares.Localidad != 'Villa Botijo'
    GROUP BY Empleados.Nombre;
'''
crsr.execute(query)
pd.read_sql_query(query, connection)

,Nombre_Empleado,Numero_Viajes
0,Prudencio Caminero,1
1,Valentin Siempre,2
2,Vicente Merario,2


In [23]:
# 7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado
query = '''
    SELECT Bares.Nombre AS Nombre_Bar, Bares.Localidad, SUM(Cervezas.Capacidad * Reparto.Cantidad) AS Total_Litros
    FROM Bares
    JOIN Reparto ON Bares.CodB = Reparto.CodB
    JOIN Cervezas ON Reparto.CodC = Cervezas.CodC
    GROUP BY Bares.Nombre, Bares.Localidad
    ORDER BY Total_Litros DESC
    LIMIT 1;
'''
crsr.execute(query)
pd.read_sql_query(query, connection)

,Nombre_Bar,Localidad,Total_Litros
0,Otra Ronda,La Esponja,359.76


In [25]:
# 8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y capacidad menor que 1 litro
query = '''
    SELECT DISTINCT Bares.Nombre AS Nombre_Bar
    FROM Bares
    JOIN Reparto ON Bares.Codb = Reparto.Codb
    JOIN Cervezas ON cervezas.CodC = Reparto.CodC
    WHERE Cervezas.envase = 'Botella' AND Cervezas.Capacidad < 1.0;
'''
crsr.execute(query)
pd.read_sql_query(query, connection)

,Nombre_Bar
0,Stop
1,Otra Ronda


In [26]:
# 9. Subir un 5% el sueldo del empleado que más días haya trabajado
#Primero obtengo el nombre del trabajador que más días ha trabajado y su sueldo
query = '''
    SELECT Empleados.Nombre, Empleados.Sueldo, COUNT(DISTINCT Reparto.Fecha) AS Dias_Trabajados
    FROM Empleados
    JOIN Reparto ON Empleados.CodE = Reparto.CodE
    GROUP BY Empleados.Nombre, Empleados.Sueldo 
    ORDER BY Dias_Trabajados DESC
    LIMIT 1;
'''
crsr.execute(query)
pd.read_sql_query(query, connection)

,Nombre,Sueldo,Dias_Trabajados
0,Vicente Merario,110000,3


In [27]:
#Ahora actualizo el sueldo del empleado que más días ha trabajado
query = '''
    UPDATE Empleados
    SET Sueldo = Sueldo * 1.05
    WHERE Nombre = 'Vicente Merario';
''' 
crsr.execute(query);

In [28]:
#Ahora muestro el sueldo actualizado del empleado que más días ha trabajado
query = '''
    SELECT Nombre, Sueldo
    FROM Empleados
    WHERE Nombre = 'Vicente Merario';
'''
crsr.execute(query)
pd.read_sql_query(query, connection)

,Nombre,Sueldo
0,Vicente Merario,115500


In [29]:
# 10. Insertar un nuevo reparto del empleado 'Vicente Merario' al bar 'Stop' de 48 cervezas de tipo lata el día 10/26/05
query = '''
    INSERT INTO Reparto (CodE, CodB, CodC, Fecha, Cantidad)
    VALUES (
        (SELECT CodE FROM Empleados WHERE Empleados.Nombre = 'Vicente Merario'),
        (SELECT CodB FROM Bares WHERE Bares.Nombre = 'Stop'),
        (SELECT CodC FROM Cervezas WHERE Cervezas.Envase = 'Lata'),
        '2005-10-26',
        48
    )
'''
crsr.execute(query)
connection.commit()

In [30]:
query = '''
    SELECT *
    FROM Reparto
''' 
crsr.execute(query)
pd.read_sql_query(query, connection)

,CodE,CodB,CodC,Fecha,Cantidad
0,1,001,01,2005-10-21,240
1,1,001,02,2005-10-21,48
2,1,002,03,2005-10-22,60
3,1,004,05,2005-10-22,4
4,2,002,03,2005-10-22,48
5,2,002,05,2005-10-23,2
6,2,004,01,2005-10-23,480
7,2,004,02,2005-10-24,72
8,3,003,03,2005-10-24,48
9,3,003,04,2005-10-25,20
